In [1]:
import numpy as np
import matplotlib.pylab as plt

# Lab 3: Homework

## Assignments

- **task1.py: MatrixProfile**
    - Improve the implementation of 
MatrixProfile with better exclusion
    - Use the MatrixProfile implementation to detect Discords

- **task2.py: Dynamic Time Warping**
    - Finalize the generic DTW implementation
    - Implement ADTW and WDTW in terms of the generic implementation

- **task3.py: Forecasting**
    - Extend the iterative forecast method to support (constant) exogenous variables.

## Task 1: MatrixProfile

### Improve MatrixProfile with better exclusion

The MatrixProfile helps identify patterns and motifs in a time series by computing the minimum distance between each subsequence and its nearest non-trivial match. However, trivial matches (where a subsequence is most similar to itself or its immediate neighbors) can distort results.

To address this, we introduce an exclusion zone, a region around the query that is ignored when searching for the nearest match.

The exclusion zone is controlled by the exclude parameter, a float between 0 and 1:

* `exclude = 0.0`: Only the exact index of the query is excluded.
* `exclude = 1.0`: The exclusion region spans `len(query)//2` indices on both sides of the query.

The exclusion size is computed as:


`exclusion_size = (len(query) * exclude)//2`

#### Examples
Assume we set `exclude=1` for the following.

```python
ts = [1,2,3,4,5,6,7,8]
query = ts[0:3]  # [1,2,3], length = 3
```

With exclude=1.0, we exclude `exclusion_size = (3 * 1.0) // 2 = 1` on both sides, meaning that the subsequences `[1,2,3]` (index 0) and `[2,3,4]` (index 1) are excluded.

```python
query = ts[2:5]  # [3,4,5], length = 3
```

Again, `exclusion_size = 1`, so we exclude:

* `[2,3,4]` (index 1), `[3,4,5]` (index 2), and `[4,5,6]` (index 3).

Change the files in `task1.py` (functions are imported below)

In [2]:
from task1 import matrix_profile

### Detect Discords

You have implemented the `matrix_profile(ts, window)`, which returns:
* mp: The Matrix Profile (distance to the nearest neighbor for each subsequence).
* mp_idx: The indices of the nearest neighbors.

Your task:
1. Implement the function `find_discords(ts, window, k)` which returns the indices of the top-k discords.
2. **A discord is a subsequence with the highest Matrix Profile value.**
3. To avoid redundant discords, once a discord is selected, ignore all subsequences within the window size of the selected discord (exclusion zone). That is, we should exclude from the start position up until the size of the window.


In [ ]:
from task1 import find_discords

#### Test and debug code for find_discords

In [ ]:
np.random.seed(42)
ts = np.sin(np.linspace(0, 10, 200)) + np.random.normal(0, 0.1, 200)
window = 15
k = 3

discords = find_discords(ts, window, k)

In [ ]:
mp, _ = matrix_profile(ts, window)
fig, ax = plt.subplots(2)
ax[0].plot(ts)
ax[0].axvline(discords[0], 0, 1, color="k", lw=0.5)
ax[0].axvline(discords[1], 0, 1, color="k", lw=0.5)
ax[0].axvline(discords[2], 0, 1, color="k", lw=0.5)
ax[1].plot(mp)
ax[1].axvline(discords[0], 0, 1, color="k", lw=0.5)
ax[1].axvline(discords[1], 0, 1, color="k", lw=0.5)
ax[1].axvline(discords[2], 0, 1, color="k", lw=0.5)


## Task 2: Dynamic Time Warping

You are given an incomplete implementation of an elastic distance function (corrected, as seen in the lecture) that computes an alignment cost between two sequences. Your task is to complete this function by implementing two missing components:

* `init_border`: Initializes the first row and column of the cost matrix.
* `compute_cost`: Computes the local cost between two elements of the sequences.

Your implementations should ensure that the function correctly computes

* Dynamic Time Warping
    * `init_border_dtw`
    * `compute_cost_dtw`

* Weighted Dynamic Time Warping
    * `WDTW.init_border_wdtw` (note a that it's class to initialize W)
    * `WDTW.compute_cost_wdtw`

Change the files in `task2.py` (functions are imported below)

In [ ]:
from task2 import elastic

### Dynamic time warping

In [ ]:
from task2 import compute_cost_dtw, init_border_dtw

### Weighed Dynamic Time Warping

In [ ]:
from task2 import WDTW

### Test your implementation

In [ ]:
random_state = np.random.RandomState(1)
x = random_state.randn(10)
y = random_state.randn(5)
v = elastic(x, y, init_border=init_border_dtw, compute_cost=compute_cost_dtw)
np.sqrt(v) # 3.789...

In [ ]:
random_state = np.random.RandomState(1)
x = random_state.randn(10)
y = random_state.randn(5)
wdtw = WDTW(len(x), len(y), g=0.05)


v = elastic(x, y, init_border=wdtw.init_border, compute_cost=wdtw.compute_cost)
np.sqrt(v) # 2.591...

## Task 3: Forecasting

Modify the given `create_lagged_features` and `iterative_forecast` functions to properly handle constant exogenous variables. Exogenous variables are external factors that do not depend on the predicted variable but influence it.

- `exogenous` is an array of `(timestep, n_variables)` meaning that there are `n_variables` per timestep.
- Note that you should merge the lagged data with all exogenous variables at that time step
- For iterative forecast, we always use the last know exogenous variable (it's not updated)

Change the files in `task3.py` (functions are imported below)

In [ ]:
from task3 import create_lagged_features, iterative_forecast

In [ ]:
random_state = np.random.RandomState(5)

n_samples = 100
lag = 3
ts = np.cumsum(random_state.randn(n_samples))

exogenous = np.sin(np.linspace(0, 10, n_samples - lag)).reshape(-1, 1)

train_size = int(len(ts) * 0.9)
train, test, exogenous_train = ts[:train_size], ts[train_size:], exogenous[:train_size]

x, y = create_lagged_features(train, exogenous_train, lag=lag)


In [ ]:
from sklearn.tree import DecisionTreeRegressor
reg = DecisionTreeRegressor(max_depth=25, random_state=1)
reg.fit(x, y)

In [ ]:
forecast = iterative_forecast(reg, x[-1, :lag], exogenous_train[-1], len(test))

In [ ]:
plt.plot(np.hstack([train, forecast]), label="Forecast")
plt.plot(np.hstack([train, test]), label="Test")
plt.legend()